# ⚛️ qAIR-vNext **v44** — the whole project, end to end

One notebook covering every stage: cache → data → model → sanity checks →
training → evaluation → **verification** → baselines → ablations →
visualisation.

Capped at **800 train / 200 validation** so the full loop runs in about
an hour instead of a day. That is enough to answer *"does the system
work?"* — it is **not** enough to produce a reportable accuracy number
(200 validation examples ⇒ standard error ≈ 3.1 points).

---

## ⚠️ Read this before trusting any number below

A v44 audit found that the three mechanisms making up this project's
research contribution — the **quantum circuit**, **multi-hypothesis
superposition**, and **Born-rule collapse** — were *all provably inert*,
while the model reported a plausible-looking 33.7%.

| # | Defect | Evidence |
|---|---|---|
| F1 | Circuit ignored its input — `Hadamard` prepares \|+⟩, an X-eigenstate, and `AngleEmbedding` defaults to RX ⇒ global phase only | output std across inputs = **0.000e+00** |
| F2 | Reasoner collapsed to a constant fixed point | pairwise cos → **1.000000**, input-independent |
| F3 | Prediction unchanged under *every* corruption of `H` | zeros / noise / shuffled / other question ⇒ all **294/869** |
| F4 | A third of cached hypotheses were template noise, misaligned to options | **33.1%** fallbacks; `argmax diag(H·O)` = 0.267 = chance |
| F5 | The question was never encoded | +**12.6 pts** once it was (0.3585 → 0.4849) |

**Accuracy could not have revealed any of these.** That is why
[section 9](#9) exists and why it matters more than the accuracy number.

**Every pre-v44 quantum/ablation result is void.** Do not compare
anything here against older notebooks.

---

### Cost warning

- **§10 baselines** — the direct-LLM arm runs the LLM over the validation
  set. Minutes.
- **§11 ablation grid** — 8 configs × 5 seeds = **40 training runs**.
  Days on CPU. Gated behind a flag; leave it off for a first pass.

Everything else runs comfortably.

<a id="1"></a>
## 1. Environment

### 1a. Run from the repo root

Every `from models…` / `from training…` import and every relative path in
`config.py` (`./cache`, `./ckpt`) assumes the repo root is the working
directory. Jupyter defaults the kernel cwd to the notebook's own folder.

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working directory:", os.getcwd())
assert os.path.exists("config.py"), (
    "Not at the repo root -- adjust the os.chdir(...) above manually, "
    r"e.g. os.chdir(r'D:\WORK\Research\qAIR-CSE499B')"
)

### 1b. Force Hugging Face offline

Qwen2.5-0.5B-Instruct and the MiniLM encoder are already cached locally.
Without this, `huggingface_hub` makes a network round-trip on every load
to revalidate them — and on this machine that request hangs indefinitely
(the process sits at 0% CPU, blocked on I/O, rather than failing fast).

In [ ]:
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
print("HF_HUB_OFFLINE:", os.environ["HF_HUB_OFFLINE"])

### 1c. Imports, seed, device

`set_seed` existed for this project's entire history and was **never
called from any entry point**, so no run was ever reproducible —
`collate_fn` draws a fresh `randperm` per batch, dropout is live, and
generation samples. It is wired into every entry point now, and here.

In [ ]:
from functools import partial

import torch
from torch.utils.data import DataLoader

from config import (
    CACHE_DIR, CKPT_DIR, PATIENCE, PERSISTENT_STEPS, N_QUBITS,
    BATCH_SIZE, WEIGHT_DECAY, SEED, EMBEDDING_DIM as DIM, resolve_device,
)
from training.seed import set_seed, seeded_generator
from training.dataset import QAIRDataset, collate_fn
from training.train import Trainer
from training.checkpoint import load_or_resume
from training.evaluate import evaluate
from models.full_model import QAIRvNext

set_seed(SEED)
device = resolve_device()

print("device        :", device)
print("embedding dim :", DIM)
print("n_qubits      :", N_QUBITS)

<a id="2"></a>
## 2. Run configuration

`TRAIN_SAMPLES` / `VAL_SAMPLES` are the important knobs.

> **These caps must stay consistent everywhere.** A cache built with
> `max_samples` is stored `complete=False` *by design*, so a later full
> build can resume and finish the split. The consequence: passing
> `max_samples=None` against a capped cache does **not** read 800
> samples — it starts generating the other 2,570. Every dataset call in
> this notebook therefore passes the caps explicitly.

In [ ]:
RUN_NAME = "qair_v44"

TRAIN_SAMPLES = 800     # cap -- must match how the cache was built
VAL_SAMPLES = 200

EPOCHS = 12             # 800 samples converges fast; 30 buys little here
persistent_steps = PERSISTENT_STEPS
n_qubits = N_QUBITS

# The validator-feedback pass runs the reasoner AND the quantum layer
# twice per forward. The CPU-bound circuit dominates, so turning this off
# roughly halves epoch time while iterating.
VALIDATOR_FEEDBACK = True

print(f"train/val  : {TRAIN_SAMPLES}/{VAL_SAMPLES}")
print(f"epochs     : {EPOCHS}   patience={PATIENCE}")
print(f"steps      : {persistent_steps}   n_qubits={n_qubits}")
print(f"batch      : {BATCH_SIZE}   wd={WEIGHT_DECAY}   seed={SEED}")

<a id="3"></a>
## 3. Cache

The slow one-time step: for every question the LLM writes one hypothesis
per answer option, then everything is embedded and cached.

**Build it from a terminal, not from here** — it takes ~1.5 h for these
caps on CPU and a notebook is a poor place to host a long job:

```bash
python scripts/build_cache.py --splits train validation --limit 800 200
```

It is resumable (autosaves every 50 samples), saves are atomic
(temp-file + `fsync` + rename), and a lock file stops a second build from
racing the first. The cell below only **loads** what that produced.

Since v44 each hypothesis comes from **its own completion**, one per
`(question, option)` pair. Splitting a single free-form completion on
newlines never guaranteed line *k* described option *k*, and the
resulting misalignment was silent — it is what drove `argmax diag(H·O)`
down to chance.

In [ ]:
train_ds = QAIRDataset(split="train", max_samples=TRAIN_SAMPLES, cache_dir=CACHE_DIR)
val_ds = QAIRDataset(split="validation", max_samples=VAL_SAMPLES, cache_dir=CACHE_DIR)

print(f"\nTrain samples: {len(train_ds)}")
print(f"Val samples:   {len(val_ds)}")

assert len(train_ds) > 0 and len(val_ds) > 0, "Empty dataset -- build the cache first."

### 3b. Inspect one sample

Check the fallback flags. A fallback carries **no information** about
which option is correct — before v44 these were 33% of all hypotheses and
the template embedded the option text verbatim, making noise maximally
similar to its own option, which is exactly the signal the selector keys
on. Post-fix this should read 0%.

In [ ]:
s = train_ds[0]

print("Question:", s["question"])
print()
flags = s.get("is_fallback", torch.zeros(len(s["hypotheses"]), dtype=torch.bool))
for i, (h, o) in enumerate(zip(s["hypotheses"], s["options"])):
    mark = "*" if i == s["y"] else " "
    fb = "  [FALLBACK]" if bool(flags[i]) else ""
    print(f" {mark}[{i}] OPT: {o}{fb}")
    print(f"      HYP: {h}")

print()
print("Q:", tuple(s["Q"].shape), " H:", tuple(s["H"].shape), " O:", tuple(s["O"].shape))
print("correct index y:", s["y"])

<a id="4"></a>
## 4. DataLoaders

`collate_fn` zero-pads ragged option/hypothesis counts (ARC has a few 3-
and 5-option questions) and emits `H_mask`/`O_mask`. Those masks were
computed and then **never passed to the model**, so padded options took
part in every reduction and could be returned as the prediction. They are
threaded end to end now.

In [ ]:
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    generator=seeded_generator(SEED),      # batch ORDER reproducible too
    collate_fn=partial(collate_fn, shuffle_options=True),
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=partial(collate_fn, shuffle_options=False),
)

print(f"train batches/epoch: {len(train_loader)}")
print(f"val batches:         {len(val_loader)}")

batch = next(iter(train_loader))
print()
for k, v in batch.items():
    print(f"  {k:12s} {tuple(v.shape)}  {v.dtype}")

<a id="5"></a>
## 5. Model

```
H ─▶ PersistentReasoner ─▶ Quantum/Classical layer ─▶ Validator ─┐
                                    │                            │
Q,O ────────────────────────────────┼──▶ EnergyAnswerSelector    │ potential
                                    ▼                            │ (closed loop)
                              EnergyFusion ─▶ CollapseController ─┘
                                    ▼
                    marginalise over hypotheses ─▶ scores
```

`backend` selects what occupies the quantum slot: `"quantum"` (PennyLane
circuit), `"classical_control"` (parameter-matched MLP — the control that
makes a quantum claim falsifiable), or `"none"`.

In [ ]:
model = QAIRvNext(
    dim=DIM,
    use_quantum=True,
    use_validator=True,
    persistent_steps=persistent_steps,
    n_qubits=n_qubits,
    use_question=True,          # F5 fix -- worth +12.6 pts on a probe
    validator_feedback=VALIDATOR_FEEDBACK,
    verbose=True,
).to(device)

total = sum(p.numel() for p in model.parameters())
circuit = sum(p.numel() for p in model.quantum.quantum.parameters()) \
    if hasattr(model, "quantum") and hasattr(model.quantum, "quantum") else 0

print(f"\ntotal parameters : {total:,}")
print(f"quantum circuit  : {circuit:,}  ({100*circuit/total:.4f}% of the model)")
print(f"backend          : {model.backend}")
print(f"device           : {next(model.parameters()).device}")

<a id="6"></a>
## 6. Sanity checks — *before* spending an hour training

Each of these targets a specific defect the audit found. Running them
first means a broken build fails in seconds rather than after training.

In [ ]:
import torch.nn.functional as F

ok = True

# ---- F1: does the circuit respond to its input at all? ----------------
if model.backend == "quantum":
    with torch.no_grad():
        a, _ = model.quantum(torch.randn(1, 4, DIM, device=device) * 0.05)
        b, _ = model.quantum(torch.randn(1, 4, DIM, device=device) * 0.05)
    delta = (a - b).abs().max().item()
    good = delta > 1e-6
    ok &= good
    print(f"[F1] circuit sensitivity to input : {delta:.4e}  "
          f"{'PASS' if good else 'FAIL - circuit is a constant!'}")

# ---- F2: does the reasoner keep hypotheses distinct? ------------------
def pairwise(M):
    M = F.normalize(M, dim=-1)
    S = M @ M.transpose(1, 2)
    k = M.shape[1]
    return ((S.sum((1, 2)) - S.diagonal(dim1=1, dim2=2).sum(1)) / (k * k - k)).mean().item()

Hin = torch.stack([train_ds[i]["H"] for i in range(32)]).float().to(device)
with torch.no_grad():
    Hout = model.reasoner(Hin)[0]
before, after = pairwise(Hin), pairwise(Hout)
good = after < 0.99
ok &= good
print(f"[F2] pairwise cos  in={before:.4f} -> out={after:.4f}          "
      f"{'PASS' if good else 'FAIL - hypotheses collapsed'}")
print(f"     dt={torch.sigmoid(model.reasoner.dt).item():.4f} (intended 0.10)  "
      f"decay={torch.sigmoid(model.reasoner.memory_decay).item():.4f} (0.80)")

# ---- masks: padded slots must not influence real scores --------------
# ARC is almost all 4-option, so a random batch often contains no padding
# at all and the test would pass vacuously. Append a synthetic 3-option
# sample so there is guaranteed padding to poison.
b = next(iter(val_loader))
H, O, Q = b["H"].to(device), b["O"].to(device), b["Q"].to(device)
Hm, Om = b["H_mask"].to(device), b["O_mask"].to(device)

if Om.all():
    Om = torch.cat([Om, Om[:1].clone()], 0);  Om[-1, -1] = False
    Hm = torch.cat([Hm, Hm[:1].clone()], 0);  Hm[-1, -1] = False
    H = torch.cat([H, H[:1]], 0); O = torch.cat([O, O[:1]], 0)
    Q = torch.cat([Q, Q[:1]], 0)
    print("[mask] batch had no padding; appended a synthetic 3-option sample")

model.eval()
with torch.no_grad():
    s1 = model(H, O, Q=Q, H_mask=Hm, O_mask=Om)["scores"]
    O2 = O.clone(); O2[~Om] = torch.randn_like(O2[~Om]) * 99      # poison padding
    s2 = model(H, O2, Q=Q, H_mask=Hm, O_mask=Om)["scores"]

delta = (s1 - s2).masked_select(Om).abs().max().item()
picked_pad = (~Om.gather(1, s1.argmax(1, keepdim=True))).any().item()
good = delta < 1e-5 and not picked_pad
ok &= good
print(f"[mask] {int((~Om).sum())} padded slots; poisoning them moves real "
      f"scores by {delta:.2e}; padded option predicted: {picked_pad}  "
      f"{'PASS' if good else 'FAIL - padding leaks'}")

# ---- gradients flow ---------------------------------------------------
from training.losses import compute_loss
model.train()
out = model(H, O, Q=Q, y=b["y"].to(device), H_mask=Hm, O_mask=Om)
loss = compute_loss(out, b["y"].to(device))
loss.backward()
gnorm = sum(p.grad.norm().item() ** 2 for p in model.parameters()
            if p.grad is not None) ** 0.5
model.zero_grad(set_to_none=True)
good = torch.isfinite(loss) and gnorm > 0
ok &= good
print(f"[grad] loss={loss.item():.4f}  |grad|={gnorm:.3f}  "
      f"{'PASS' if good else 'FAIL'}")

print("\n" + ("ALL SANITY CHECKS PASSED" if ok else "*** SANITY CHECKS FAILED ***"))

<a id="7"></a>
## 7. Train

Interrupting is safe — a `_latest.pt` checkpoint is written every epoch,
so re-running the Trainer cell resumes from the last completed epoch.

**Watch `Pairwise Cos` and `H Cos`, not just accuracy.** They measure
whether hypotheses stay distinguishable after reasoning. Near 1.0 means
they have merged, the collapse distribution is uniform by construction,
and the multi-hypothesis mechanism is inert — precisely what happened
pre-v44 while accuracy read a respectable 33.7%. The trainer prints a
`[COLLAPSE WARNING]` above 0.99 and names which failure mode it is:

| | |
|---|---|
| `h_cos ≈ 1` and `pairwise_cos ≈ 1` | the **reasoner** merged the hypotheses — architectural |
| `h_cos < 1` but `pairwise_cos ≈ 1` | hypotheses stay distinct but the **selector ignores them** — a low-signal cache; regenerate, don't tune |

In [ ]:
trainer = Trainer(
    model=model, train_loader=train_loader, val_loader=val_loader,
    device=device, ckpt_dir=CKPT_DIR, name=RUN_NAME,
    weight_decay=WEIGHT_DECAY, verbose=True,
)

start_epoch, best_acc, _ = load_or_resume(trainer, CKPT_DIR, RUN_NAME, EPOCHS)
print(f"\nstarting from epoch {start_epoch}/{EPOCHS}  (best so far {best_acc:.4f})")

In [ ]:
history = trainer.train(
    epochs=EPOCHS, start_epoch=start_epoch,
    best_acc=best_acc, patience=PATIENCE,
)

In [ ]:
torch.save(
    {
        "use_quantum": True, "use_validator": True,
        "persistent_steps": persistent_steps, "n_qubits": n_qubits,
        "use_question": True, "seed": SEED,
        "train_samples": TRAIN_SAMPLES, "val_samples": VAL_SAMPLES,
    },
    os.path.join(CKPT_DIR, f"{RUN_NAME}_config.pt"),
)
print("[CONFIG SAVED]")

### 7b. Training curves

In [ ]:
import matplotlib.pyplot as plt

EXPORT_DIR = "./exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(history["loss"]);  ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(history["acc"]);   ax[1].axhline(0.25, ls="--", c="r", label="chance")
ax[1].set_title("val accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
ax[2].plot(history["pairwise_cos"], label="pairwise_cos (energy rows)")
ax[2].plot(history["h_cos"], label="h_cos (hypothesis vectors)")
ax[2].axhline(0.99, ls="--", c="r", label="collapse")
ax[2].set_ylim(0, 1.05); ax[2].set_title("hypothesis distinguishability")
ax[2].set_xlabel("epoch"); ax[2].legend()
plt.tight_layout()
plt.savefig(f"{EXPORT_DIR}/{RUN_NAME}_curves.png", bbox_inches="tight")
plt.show()

print(f"best val acc : {history['best_acc']:.4f}   (chance = 0.2500)")
print(f"final pairwise_cos : {history['pairwise_cos'][-1]:.4f}")
print(f"final h_cos        : {history['h_cos'][-1]:.4f}")

<a id="8"></a>
## 8. Evaluation

Reload the **best** checkpoint rather than evaluating whatever weights
the last epoch happened to leave behind.

In [ ]:
best_path = os.path.join(CKPT_DIR, f"{RUN_NAME}_best.pt")
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=device)["model"])
    print(f"[LOADED] {best_path}")

eval_model = model
metrics = evaluate(eval_model, val_loader, device)

for k, v in metrics.items():
    print(f"  {k:15s}: {v:.4f}")

print(f"\n  n = {len(val_ds)} examples, so the standard error is roughly "
      f"{(0.25*0.75/len(val_ds))**0.5:.3f} -- treat this accuracy as "
      f"'above chance or not', nothing finer.")

<a id="9"></a>
## 9. ⭐ Input ablation — does the model actually use its hypotheses?

**The most important cell in this notebook.**

qAIR claims to reason over multiple hypotheses held in superposition.
That claim is falsifiable in one line: corrupt the hypotheses and see
whether the answer changes. Pre-v44 it did not — not "changed a little",
but **bit-identically unchanged** at 294/869 under every corruption
tried:

```
intact H, O                   294/869 = 0.3383
H := zeros                    294/869 = 0.3383
H := gaussian noise           294/869 = 0.3383
H shuffled within sample      294/869 = 0.3383
H from a DIFFERENT question   294/869 = 0.3383
H := O (option echo)          294/869 = 0.3383
O := zeros                    223/869 = 0.2566   ← only O ever mattered
```

Accuracy alone could never have shown that. **If this reports FAIL, no
number in this notebook supports any claim about superposition, collapse,
or quantum reasoning** — however good it looks.

In [ ]:
from evaluation.input_ablation import input_ablation, report

ab_results = input_ablation(eval_model, val_loader, device)
passed = report(ab_results)

<a id="10"></a>
## 10. Reference baselines

Two numbers that decide how to read everything above.

**Data ceiling** — probes straight on the cached embeddings. No
architecture over this cache can do much better, so a qAIR number below
these is a problem with qAIR, not with the task. Measured pre-fix on the
full validation split:

| probe | val acc |
|---|---|
| O only | 0.3585 |
| diag(H_n, O_n) | 0.3527 |
| full pairwise H_k × O_n | 0.3979 |
| **Q × O pairwise** | **0.4849** |
| Q × O + H | 0.4408 ← the old `H` made it *worse* |

The full 8.2M-parameter pre-fix model scored **0.3383** — below the
options-only MLP.

**Direct LLM** — scores each option's likelihood under Qwen2.5-0.5B, the
same model that generated the hypotheses. If the pipeline cannot beat the
model it is built on top of, it is a lossy compression of knowledge the
generator already had. **This is the first thing a reviewer will check.**

In [ ]:
from evaluation.baselines import run_probes

probe_results = run_probes(
    CACHE_DIR, train_samples=TRAIN_SAMPLES, val_samples=VAL_SAMPLES,
)

In [ ]:
from evaluation.baselines import run_direct

# limit=None scores every loaded validation sample. Set e.g. limit=100
# for a faster read.
direct_acc, direct_records = run_direct(
    CACHE_DIR, split="validation", limit=None, max_samples=VAL_SAMPLES,
)

print(f"\nqAIR   : {metrics['acc']:.4f}")
print(f"direct : {direct_acc:.4f}")
print("\n" + ("qAIR beats its own generator." if metrics["acc"] > direct_acc
              else "*** The generator alone beats the full pipeline. ***"))

<a id="11"></a>
## 11. Ablation grid

> ### ⚠️ Off by default — this is 40 training runs
> `run_ablation_suite` sweeps every config in `ABLATIONS` (8) against
> every seed in `config.SEEDS` (5). On CPU, with the quantum layer
> running twice per forward, that is **days**. Flip `RUN_GRID` only when
> you can afford it — ideally on a GPU.

The load-bearing comparison is **`A1b_quantum_only` vs
`A1c_classical_control_only`**. `ClassicalControlLayer` reproduces the
quantum layer's entire surrounding architecture — same compress/expand
widths, same fusion gate, same correction and energy heads, same input
squashing — and swaps *only* the PennyLane circuit for a classical MLP at
the circuit's measurement width. Total parameters match to within a
fraction of a percent (the circuit is 54 of ~1.12M), so a gap cannot be
blamed on capacity. It is the circuit, or it is nothing.

Single-run numbers are not interpretable: even the full 869-example
validation split gives a standard error of ~1.6 points, larger than most
gaps this grid produces. Hence multi-seed + the paired tests in §12.

In [ ]:
from training.ablations import run_ablation_suite, ABLATIONS

RUN_GRID = False        # ⚠️ True = 8 configs x N seeds of training

print("configs in the grid:")
for name, cfg in ABLATIONS.items():
    print(f"  {name:<34s} {cfg}")

if RUN_GRID:
    ablation_results = run_ablation_suite(
        cache_dir=CACHE_DIR, ckpt_dir=CKPT_DIR,
        epochs=EPOCHS, patience=PATIENCE, n_qubits=n_qubits,
        seeds=(SEED,),          # start with one seed
        with_test=False,        # no test cache built at these caps
    )
else:
    ablation_results = None
    print("\nSkipped (RUN_GRID=False).")

<a id="12"></a>
## 12. Statistics

Bootstrap CIs plus **paired** McNemar tests. Paired matters: two arms see
the same examples, so testing the disagreement cells is far more powerful
than comparing two independent proportions.

In [ ]:
from evaluation.stats import compare_arms, bootstrap_ci

if ablation_results:
    compare_arms(ablation_results, baseline="A1_baseline")
else:
    print("No grid results -- run §11 with RUN_GRID=True first.\n")

_, recs = evaluate(eval_model, val_loader, device, return_records=True)
acc, lo, hi = bootstrap_ci(recs)
print(f"{RUN_NAME}: acc={acc:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
print(f"chance = 0.2500 -- {'ABOVE' if lo > 0.25 else 'NOT distinguishable from'} chance.")

<a id="13"></a>
## 13. Visualisation

`keep_trajectory` is off during training (it forced a device→host sync on
every reasoning step of every forward pass). Enabled just for this pass.

In [ ]:
eval_model.reasoner.keep_trajectory = True

viz_loader = DataLoader(val_ds, batch_size=1, shuffle=True,
                        collate_fn=partial(collate_fn, shuffle_options=False))
vb = next(iter(viz_loader))

with torch.no_grad():
    viz = eval_model(
        vb["H"].to(device), vb["O"].to(device), Q=vb["Q"].to(device),
        H_mask=vb["H_mask"].to(device), O_mask=vb["O_mask"].to(device),
    )

eval_model.reasoner.keep_trajectory = False

probs = viz["collapse_probs"][0].cpu().numpy()
print("collapse_probs:", probs.round(4))
print(f"uniform would be {1/len(probs):.4f} -- "
      f"{'DEGENERATE' if abs(probs.max() - 1/len(probs)) < 0.01 else 'non-uniform, good'}")

In [ ]:
from visualization.attention_maps import plot_attention_map
from visualization.energy_maps import plot_energy_map

att = viz["attention"]
att = att[0, 0] if att.dim() == 4 else att[0]
plot_attention_map(att, save_path=f"{EXPORT_DIR}/{RUN_NAME}_attention.png")
plot_energy_map(viz["answer_energy"][0], save_path=f"{EXPORT_DIR}/{RUN_NAME}_energy.png")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(range(len(probs)), probs)
ax[0].axhline(1 / len(probs), ls="--", c="r", label="uniform (= inert)")
ax[0].set_xlabel("hypothesis"); ax[0].set_ylabel("P"); ax[0].legend()
ax[0].set_title("collapse distribution")

v = viz["validator"]
if v is not None:
    names = ["causal", "diversity", "specificity", "relevance"]
    ax[1].bar(names, [v[n][0].mean().item() for n in names])
    ax[1].set_title("validator observables")
plt.tight_layout()
plt.savefig(f"{EXPORT_DIR}/{RUN_NAME}_collapse.png", bbox_inches="tight")
plt.show()

In [ ]:
from visualization.trajectory import plot_trajectory

if viz["trajectory"]:
    plot_trajectory(viz["trajectory"], save_path=f"{EXPORT_DIR}/{RUN_NAME}_trajectory.png")
else:
    print("no trajectory captured")

<a id="14"></a>
## 14. Summary

Reads the run back and states plainly what it does and does not support.

In [ ]:
print("=" * 68)
print(f"qAIR-vNext v44  --  {RUN_NAME}")
print("=" * 68)
print(f"data          : {len(train_ds)} train / {len(val_ds)} val  (capped)")
print(f"backend       : {model.backend}   question={model.use_question}"
      f"   feedback={model.validator_feedback}")
print(f"parameters    : {total:,}  (circuit {circuit:,})")
print()
print(f"val accuracy  : {metrics['acc']:.4f}   [95% CI {lo:.4f}, {hi:.4f}]")
print(f"chance        : 0.2500")
print(f"direct LLM    : {direct_acc:.4f}" if 'direct_acc' in dir() else "direct LLM    : not run")
print()
print(f"pairwise_cos  : {metrics['pairwise_cos']:.4f}  (1.0 = inert)")
print(f"h_cos         : {metrics['h_cos']:.4f}")
print(f"collapse_peak : {metrics['collapse_peak']:.4f}  (1/K = uniform = inert)")
print()
print(f"input ablation: {'PASS -- hypotheses matter' if passed else 'FAIL -- hypotheses unused'}")
print("=" * 68)

if not passed:
    print(
        "\nThe multi-hypothesis mechanism is inert in this run.\n"
        "Report nothing about superposition, collapse or quantum reasoning\n"
        "from this checkpoint. Check the cache fallback rate first (§3b),\n"
        "then h_cos vs pairwise_cos above to tell reasoner-collapse from\n"
        "selector-indifference."
    )
elif lo <= 0.25:
    print("\nAccuracy is not distinguishable from chance at this sample size.")

---

### Next steps

1. **Scale the cache up.** 800/200 answers "does it work", not "how well".
   ```bash
   python scripts/build_cache.py --splits train validation
   ```
2. **Build the test split** — validation currently does double duty as
   both model selection and reported metric, which biases it upward.
3. **Run the grid** (§11) with multiple seeds, on a GPU.
4. **Complex amplitudes.** With real, non-negative amplitudes the
   Born-rule step is *algebraically identical* to `softmax(−E/T)`
   (verified to fp32 epsilon). Until interference cross-terms
   `2·Re(aᵢ·conj(aⱼ))` actually contribute, it is notation rather than
   physics — and a write-up must say so.